In [1]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
!git log --oneline -3

!pip install -q peft accelerate openai

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json, yaml, torch, sys
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))   # set in Colab: key icon (Secrets) -> HF_TOKEN

sys.path.insert(0, "src/evaluation")
import eval as cpi_eval

cfg = yaml.safe_load(open("src/mechanistic-analysis/checkpoints.yaml"))

tok = AutoTokenizer.from_pretrained(cfg["base_model"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

tok.chat_template = (
    "{{ bos_token }} "
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "{{ '<|system|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'user' %}"
    "{{ '<|user|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|assistant|>\n' + message['content'].strip() + eos_token + '\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}"
)
USE_CT = True

ref = json.load(open("results/cpi-results/tulu-results/checkpoint_step3150_metrics.json",
                     encoding="utf-8"))
row = next(r for r in ref["per_item"] if r["item_id"] == "cap_0013")
ours = tok.decode(cpi_eval.build_score_prompt_ids(tok, row["question"], row["context"], USE_CT),
                  skip_special_tokens=False)
assert ours == row["score_prompt_text"], f"PROMPT MISMATCH\nours  : {ours!r}\nstored: {row['score_prompt_text']!r}"
print("prompt verified")

n = len(tok(" Kabul", add_special_tokens=False).input_ids)
assert n == 2, f"expected 2 tokens for ' Kabul', got {n} -- double-space bug is back"
print("tokenisation verified")

base_model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"], torch_dtype=torch.bfloat16, device_map="cuda").eval()
print("base model loaded")

In [ ]:
import re, glob, time, shutil

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")   # Colab Secrets -> OPENAI_API_KEY

STAGE   = "/content/_ckpt"
OUT_DIR = {"alpaca": "results/cpi-results/alpaca-results",
           "tulu":   "results/cpi-results/tulu-results"}

items = cpi_eval.load_items("dataset/conflict_eval_unified.json")
pool  = cpi_eval.build_distractor_pool(items)
judge = cpi_eval.make_judge(prompt_version="v2")     # used for EVERY checkpoint
print("judge ready")

_cur = {"m": base_model}
def swap(path):
    m = _cur["m"]
    if isinstance(m, PeftModel):
        m = m.unload()
    if path is not None:
        m = PeftModel.from_pretrained(m, path)
    m.eval()
    _cur["m"] = m
    return m


def drive_ckpts(run):
    """{step: path} for every adapter on Drive."""
    d = cfg["runs"][run]["drive_checkpoint_dir"]
    out = {}
    for p in glob.glob(os.path.join(d, "checkpoint-*")):
        m = re.search(r"checkpoint-(\d+)$", p.replace("\\", "/"))
        if m:
            out[int(m.group(1))] = p
    return out


def score_one(run, step, path, out_dir):
    """Stage -> attach -> full eval (filter/generate/logprob/judge) -> write json."""
    t0 = time.time()
    shutil.rmtree(STAGE, ignore_errors=True)
    shutil.copytree(path, STAGE)
    t_io = time.time() - t0

    model = swap(STAGE)
    cpi_eval.evaluate_checkpoint(
        model, tok, items=items, distractor_pool=pool,
        output_path=out_dir, checkpoint_id=f"step{step}", global_step=step,
        use_ct=USE_CT, judge_fn_=judge,
        methods=("paper", "ordered", "logprob"), verbose=True)

    f = f"{out_dir}/checkpoint_step{step}_metrics.json"
    d = json.load(open(f, encoding="utf-8"))
    d["rescored"] = True
    json.dump(d, open(f, "w"), indent=2)
    print(f"  step{step:<5} io={t_io:5.1f}s  total={time.time()-t0:5.1f}s", flush=True)
    return f

In [ ]:
# HISTORICAL -- already run and passed (2026-09-21). Do not re-run: it validates against
# first_token_gap.json, which was deleted in the mechanistic-analysis reset. Kept as the
# record of how the rescore was validated (exact delta match + n_par tokenisation check).
from collections import Counter

VALIDATE = {"tulu": 3150, "alpaca": 800}     # both have corrected references
TMP = "/content/_validate"
os.makedirs(TMP, exist_ok=True)

fkg = json.load(open("src/mechanistic-analysis/first_token_gap.json", encoding="utf-8"))
PHASE_OF = {("tulu", 3150): "trough", ("alpaca", 800): "trough"}

for run, step in VALIDATE.items():
    print(f"\n=== validating {run} step{step} ===", flush=True)
    f = score_one(run, step, drive_ckpts(run)[step], TMP)
    new = json.load(open(f, encoding="utf-8"))
    old = json.load(open(f"{OUT_DIR[run]}/checkpoint_step{step}_metrics.json", encoding="utf-8"))

    # 1. does the corrected delta match first_token_gap.json?
    ref = {r["item_id"]: r["delta_score"] for r in fkg
           if r["run"] == run and r["phase"] == PHASE_OF[(run, step)]}
    diffs = [r["logprob"]["delta"] - ref[r["item_id"]]
             for r in new["per_item"] if r["item_id"] in ref and r.get("logprob")]
    print(f"  delta vs first_token_gap: n={len(diffs)}  max|diff|={max(map(abs, diffs)):.4f}  (want ~0)")

    # 2. tokenisation: n_par must match a fresh single-space tokenisation
    bad = sum(1 for r in new["per_item"] if r.get("logprob") and
              r["logprob"]["n_par"] != len(tok(" " + r["parametric_answer"].strip(),
                                               add_special_tokens=False).input_ids))
    print(f"  n_par mismatches vs fresh tokenisation: {bad}  (want 0)")

    # 3. what actually changed
    def summary(d):
        p = [r for r in d["per_item"] if r.get("passed")]
        c = Counter(r["final_label"] for r in p if r.get("final_label"))
        amb = sum(1 for r in p if r.get("logprob", {}).get("label") == "AMBIG")
        n = c["CTX"] + c["PAR"]
        return len(p), c["CTX"] / max(n, 1), amb / max(len(p), 1)
    for name, d in [("old", old), ("new", new)]:
        np_, rc, am = summary(d)
        print(f"  {name}: passed={np_:4}  R_ctx={rc:.3f}  AMBIG_rate={am:.2f}")

In [ ]:
# Phase-window rescoring: instead of a full dense sweep, rescore only each
# peak/trough/recovery checkpoint plus its 2 neighbours on each side (checkpoints
# are saved every 50 steps -- see congif.yaml `checkpointing.save_steps`). That's
# enough to confirm each phase pick is still a local extremum under the corrected
# scorer, without paying for the full ~116-checkpoint sweep.
NEIGHBOR_RADIUS = 2          # checkpoints each side of a phase pick
STEP_SPACING    = 50         # save_steps in congif.yaml

PHASE = {"alpaca": [100, 800], "tulu": [500, 3150, 5000]}

def selected(run):
    found = drive_ckpts(run)
    window = set()
    for p in PHASE[run]:
        for k in range(-NEIGHBOR_RADIUS, NEIGHBOR_RADIUS + 1):
            window.add(p + k * STEP_SPACING)
    keep = window & set(found)
    return [(s, found[s]) for s in sorted(keep)]

def already(run, step):
    f = f"{OUT_DIR[run]}/checkpoint_step{step}_metrics.json"
    return os.path.exists(f) and json.load(open(f, encoding="utf-8")).get("rescored") is True

for run in ["alpaca", "tulu"]:
    todo = selected(run)
    print(f"\n=== {run}: {len(todo)} selected (phase +/- {NEIGHBOR_RADIUS} checkpoints) ===", flush=True)
    print(f"  steps: {[s for s, _ in todo]}")
    for step, path in todo:
        if already(run, step):
            print(f"  step{step:<5} skip (already rescored)")
            continue
        score_one(run, step, path, OUT_DIR[run])

shutil.rmtree(STAGE, ignore_errors=True)
print("\ndone")

In [ ]:
# One-time backup: alpaca is done, but its 7 results only exist in the ephemeral
# /content clone right now. Copy them to Drive so a runtime reset / `git reset --hard`
# in cell 0 can never wipe them. Safe to re-run (just overwrites with identical files).

DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/CPI-Analysis-Results"   # <-- change if you'd rather use an existing folder
OUT_DIR_DRIVE = {
    "alpaca": f"{DRIVE_RESULTS_ROOT}/alpaca-results",
    "tulu":   f"{DRIVE_RESULTS_ROOT}/tulu-results",
}
os.makedirs(OUT_DIR_DRIVE["alpaca"], exist_ok=True)
os.makedirs(OUT_DIR_DRIVE["tulu"], exist_ok=True)

n_backed_up = 0
for step in [50, 100, 150, 200, 700, 750, 800]:
    src = f"{OUT_DIR['alpaca']}/checkpoint_step{step}_metrics.json"
    if os.path.exists(src):
        shutil.copy2(src, OUT_DIR_DRIVE["alpaca"])
        n_backed_up += 1
print(f"backed up {n_backed_up}/7 alpaca result files to Drive: {OUT_DIR_DRIVE['alpaca']}")

In [ ]:
# TULU only: phase-window rescore (peak 500, trough 3150, recovery 5000, each +/- 2
# checkpoints) writing DIRECTLY to Drive -- OUT_DIR_DRIVE, not the local /content clone --
# so results persist through a runtime restart and resuming after a disconnect just works.

def selected_tulu():
    found = drive_ckpts("tulu")
    window = set()
    for p in PHASE["tulu"]:
        for k in range(-NEIGHBOR_RADIUS, NEIGHBOR_RADIUS + 1):
            window.add(p + k * STEP_SPACING)
    keep = window & set(found)
    return [(s, found[s]) for s in sorted(keep)]

def already_drive(run, step):
    f = f"{OUT_DIR_DRIVE[run]}/checkpoint_step{step}_metrics.json"
    return os.path.exists(f) and json.load(open(f, encoding="utf-8")).get("rescored") is True

todo = selected_tulu()
print(f"=== tulu: {len(todo)} selected (phase +/- {NEIGHBOR_RADIUS} checkpoints), writing directly to Drive ===", flush=True)
print(f"  steps: {[s for s, _ in todo]}")
for step, path in todo:
    if already_drive("tulu", step):
        print(f"  step{step:<5} skip (already rescored, found in Drive)")
        continue
    score_one("tulu", step, path, OUT_DIR_DRIVE["tulu"])

shutil.rmtree(STAGE, ignore_errors=True)
print(f"\ndone -- tulu results are in {OUT_DIR_DRIVE['tulu']} (persistent, survives runtime reset)")